In [2]:
import pandas as pd

In [3]:
#as usual we use the panda function to read the csv file into a dataset variable
dataset = pd.read_csv("insurance_pre.csv")

In [4]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [5]:
#first we convert all the nominal values into numerical values using On hot encoding
#the column names are the unique value you find in the original state
# the value is set to true(0) based on the exact value in the original set, rest of the other two column is set to false(0)
# the usage of drop_first is to avoid "Dummy Variable Trap" - information repeatition confuses machine learning algorithm at some point. 
# it means when you have perfect multicollinearity, one variable can be predicted perfectly from the others.
# in this example when California is removed, we can still guess that when florida and New York are zero, obviously california is 1. 
dataset = pd.get_dummies(dataset,dtype=int,drop_first=True)

In [6]:
#when we search the dataset again, we see that the unique values have undergone column expansion
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [7]:
#it helps us giving the column names available in the dataset variable
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [8]:
#set the input variable
independent = dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]

In [9]:
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [10]:
# set the dependent variable
dependent = dataset[['charges']]

In [11]:
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [12]:
#this block explains how to split training and test dataset separately
#just like pandas, Scikit-Learn(sklearn) is a library
#we set test_size allocation as 30% of the dataset for testing(as test data), 70% to training dataset
#Pandas is relatively a compact library and so its fast and does not waste much computer memory
#Sklearn is a gigantic library and it has lots of algorithm inside it. So your code would become slow, when you load the whole thing like pandas 
#we tell the code to go into the massive Scikit-Learn warehouse, find the model_selection folder, and grab only the train_test_split tool
#so that it doesnt occupy too much memory

from sklearn.model_selection import train_test_split
#remember x is input, y is output
#we use the variables train to use the train dataset to train the model and test variables to explicitly state which test the model 
#x and y represents input as 5 input and output as profit
#these variables individually stores all the separated data
#random_state is set to 0, so that next time this code is run the data is segregated the same way into train and test
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size=0.30, random_state=0)

In [13]:
#standardize the input
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [14]:
#this block helps the model generate the w(slope), b(intercept) in y = wx+b
#once this code runs the model has been created

#goes into Scikit-Learn's linear_model folder and grabs only the LinearRegression tool
from sklearn.linear_model import LinearRegression

#Creates the Linear Regression model and stores it in a variable named regressor. The model is absolutely empty at this point
regressor = LinearRegression(fit_intercept=False, positive= False)

#This is the actual training step. The .fit function tells the model to calculate a straight line of best fit using the training dataset.
#basically the model is getting trained using the dataset and that is when the value of slope and bias are calculated 
#and gets stored in Attributes named (1) coef_ (contains slope) and (2) intercept_ (contains bias)
regressor.fit(x_train,y_train)

,fit_intercept,False
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [15]:
#w=weight or slope. The variable is set using coef_ as said in the above cell
weight = regressor.coef_
weight

array([[3610.51960788, 2000.92823398,  565.16490411,  -20.86836183,
        9419.60171825]])

In [16]:
#b = bias or intercept or origin or startpoint
#The variable is set using intercept_ as said in the above cell
bias = regressor.intercept_
bias

0.0

In [17]:
#using built-in Scikit-Learn function predict() tells the model to calculate outputs based on the math it learned during training.
# x_test is the 30% of the test data that the model never knows till now
#y_pred a new variable that stores the model's predicted salary guesses
y_pred = regressor.predict(x_test)


In [18]:
#evaluation metrics
#recollecting the fourth validating paramer R square: we try to compare to what level the y_pred(predicted salary) and 
# y_test(salary taken from our original test dataset) matches and store that score to the variable r_score
from  sklearn.metrics import r2_score
r_score = r2_score(y_test, y_pred)

In [19]:
# the received score wud tell you how well the model is working
r_score

-0.35636266509759285